# Track B: Feature Engineering (Santander)
This notebook transforms the monthly product snapshots into a static, cross-sectional product matrix and demographic vector for the Autoencoder.


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os


## 1. Load Data with Optimized Types


In [2]:
# We define a subset of dtypes with their ORIGINAL SPANISH names for loading
dtypes = {
    'ncodpers': np.int32, 'age': object, 'antiguedad': object, 'renta': np.float32,
    'ind_nuevo': np.float32, 'indrel': np.float32, 'indrel_1mes': object, 'indext': object,
    'conyuemp': object, 'indfall': object, 'tipodom': np.float32, 'ind_actividad_cliente': np.float32,
    'segmento': object, 'sexo': object, 'tiprel_1mes': object,
    'ind_ahor_fin_ult1': np.float32, 'ind_aval_fin_ult1': np.float32,
    'ind_cco_fin_ult1': np.float32, 'ind_cder_fin_ult1': np.float32, 'ind_cno_fin_ult1': np.float32,
    'ind_ctju_fin_ult1': np.float32, 'ind_ctma_fin_ult1': np.float32, 'ind_ctop_fin_ult1': np.float32,
    'ind_ctpp_fin_ult1': np.float32, 'ind_deco_fin_ult1': np.float32, 'ind_deme_fin_ult1': np.float32,
    'ind_dela_fin_ult1': np.float32, 'ind_ecue_fin_ult1': np.float32, 'ind_fond_fin_ult1': np.float32,
    'ind_hip_fin_ult1': np.float32, 'ind_plan_fin_ult1': np.float32, 'ind_pres_fin_ult1': np.float32,
    'ind_reca_fin_ult1': np.float32, 'ind_tjcr_fin_ult1': np.float32, 'ind_valo_fin_ult1': np.float32,
    'ind_viv_fin_ult1': np.float32, 'ind_nomina_ult1': np.float32, 'ind_nom_pens_ult1': np.float32,
    'ind_recibo_ult1': np.float32, 'nomprov': str, 'fecha_dato': str
}

# Translation dictionary
rename_dict = {
    'ncodpers': 'customer_id', 'age': 'age', 'antiguedad': 'seniority_months', 'renta': 'gross_income',
    'segmento': 'customer_segment', 'sexo': 'gender', 'nomprov': 'province_name', 'fecha_dato': 'snapshot_date',
    'ind_nuevo': 'new_customer_index', 'indrel': 'customer_type', 'indrel_1mes': 'customer_type_primary',
    'indext': 'foreigner_index', 'conyuemp': 'spouse_index', 'indfall': 'deceased_index', 'tipodom': 'address_type',
    'ind_actividad_cliente': 'activity_index', 'tiprel_1mes': 'customer_relation_type',
    'ind_ahor_fin_ult1': 'has_savings_account', 'ind_aval_fin_ult1': 'has_bank_guarantees',
    'ind_cco_fin_ult1': 'has_current_account', 'ind_cder_fin_ult1': 'has_derivada_account',
    'ind_cno_fin_ult1': 'has_payroll_account', 'ind_ctju_fin_ult1': 'has_junior_account',
    'ind_ctma_fin_ult1': 'has_mas_particular_account', 'ind_ctop_fin_ult1': 'has_particular_account',
    'ind_ctpp_fin_ult1': 'has_particular_plus_account', 'ind_deco_fin_ult1': 'has_short_term_deposits',
    'ind_deme_fin_ult1': 'has_medium_term_deposits', 'ind_dela_fin_ult1': 'has_long_term_deposits',
    'ind_ecue_fin_ult1': 'has_e_account', 'ind_fond_fin_ult1': 'has_funds', 'ind_hip_fin_ult1': 'has_mortgage',
    'ind_plan_fin_ult1': 'has_pensions', 'ind_pres_fin_ult1': 'has_loans', 'ind_reca_fin_ult1': 'has_taxes',
    'ind_tjcr_fin_ult1': 'has_credit_card', 'ind_valo_fin_ult1': 'has_securities', 'ind_viv_fin_ult1': 'has_home_account',
    'ind_nomina_ult1': 'has_payroll', 'ind_nom_pens_ult1': 'has_pensions_nom', 'ind_recibo_ult1': 'has_direct_debit'
}

chunksize = 1000000
chunks = []
data_path = '../data/santander/train_ver2.csv'

print("Processing 13.6M rows in chunks to extract 2016-05-28 snapshot...")
for chunk in pd.read_csv(data_path, dtype=dtypes, usecols=list(dtypes.keys()), chunksize=chunksize):
    # Rename columns to English immediately
    chunk.rename(columns=rename_dict, inplace=True)
    # Filter immediately using the NEW English name
    filtered_chunk = chunk[chunk['snapshot_date'] == '2016-05-28']
    chunks.append(filtered_chunk)

df_current = pd.concat(chunks, ignore_index=True)
print(f"Successfully extracted {len(df_current)} records for 2016-05-28")

Processing 13.6M rows in chunks to extract 2016-05-28 snapshot...


Successfully extracted 931453 records for 2016-05-28


## 2. Temporal Filtering
Filter for the most recent month in the dataset to capture current state.


In [3]:
# Filtering was already handled during chunked loading to save memory.
print(f'Shape after temporal filtering: {df_current.shape}')

Shape after temporal filtering: (931453, 39)


## 3. Imputation and Cleaning


In [4]:
# Numeric conversions
df_current['age'] = pd.to_numeric(df_current['age'], errors='coerce')
df_current['seniority_months'] = pd.to_numeric(df_current['seniority_months'], errors='coerce') # Seniority in months

# Impute Income (gross_income) by Province (province_name) median
province_medians = df_current.groupby('province_name')['gross_income'].transform('median')
df_current['gross_income'] = df_current['gross_income'].fillna(province_medians)

# Fill remaining demographics with overall median
df_current['gross_income'] = df_current['gross_income'].fillna(df_current['gross_income'].median())
df_current['age'] = df_current['age'].fillna(df_current['age'].median())
df_current['seniority_months'] = df_current['seniority_months'].fillna(df_current['seniority_months'].median())


## 4. Categorical Encoding & Product Vectorization


In [5]:
# Find product columns (all start with 'has_')
products = [c for c in df_current.columns if c.startswith('has_')]

# One-hot encode segments
df_current = pd.get_dummies(df_current, columns=['customer_segment'], dummy_na=False, dtype=int)

# Santander sexo coding: H=female, V=male; unknown and missing remain null.
df_current['gender'] = df_current['gender'].astype('string').str.strip().str.upper()
df_current['is_female'] = df_current['gender'].map({'H': 1, 'V': 0}).astype('Int8')

# Fill missing products with 0
df_current[products] = df_current[products].fillna(0)

# Drop unneeded text columns
df_current.drop(columns=['province_name', 'snapshot_date'], inplace=True, errors='ignore')

## 5. Normalization and Saving


In [6]:
continuous_features = ['age', 'gross_income', 'seniority_months']

scaler = StandardScaler()
df_current[continuous_features] = scaler.fit_transform(df_current[continuous_features])

os.makedirs('../data/processed', exist_ok=True)
df_current.to_csv('../data/processed/track_b_features.csv', index=False)
print('Track B features successfully saved to data/processed/track_b_features.csv')
display(df_current.head())


Track B features successfully saved to data/processed/track_b_features.csv


,customer_id,age,new_customer_index,seniority_months,customer_type,customer_type_primary,customer_relation_type,foreigner_index,spouse_index,deceased_index,...,has_credit_card,has_securities,has_home_account,has_payroll,has_pensions_nom,has_direct_debit,customer_segment_01 - TOP,customer_segment_02 - PARTICULARES,customer_segment_03 - UNIVERSITARIO,is_female
0,657640,1.619497,0.0,0.020784,1.0,1,I,N,NaN,N,...,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0
1,657788,0.106585,0.0,0.020784,1.0,1,A,N,NaN,N,...,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0
2,657795,0.222963,0.0,0.020784,1.0,1,A,N,NaN,N,...,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,1
3,657790,0.106585,0.0,0.020784,1.0,1,A,N,NaN,N,...,1.0,0.0,0.0,1.0,1.0,1.0,0,1,0,1
4,657794,0.513907,0.0,0.020784,1.0,1,A,N,NaN,N,...,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0,0
